# Retail Sales Performance Analysis
**Dataset:** `01_retail_sales.csv`  
**Industry:** Retail / E-commerce

Analyses transaction records to understand revenue, product performance, customer value and regional trends. Each task from the brief has its own section below.

**Setup:** place the CSV in the same folder as this notebook (or change `DATA_PATH` in the config cell). Outputs are written to `outputs/`.

## Setup and configuration

In [33]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("retail_sales")

# --- Column names: adjust to match your CSV headers -------------------------
COL = {
    "order_id": "Order_ID",
    "date": "Order_Date",
    "customer": "Customer_ID",
    "product": "Product",
    "category": "Category",
    "city": "City",
    "payment": "Payment_Mode",
    "quantity": "Quantity",
    "unit_price": "Unit_Price",
    "revenue": "Revenue",
}

# --- Paths -------------------------------------------------------------------
DATA_PATH = Path("/content/02_retail_sales_alternative_products.csv")
OUTPUT_DIR = Path("outputs")
OUT_CLEAN = OUTPUT_DIR / "02_retail_sales_alternative_products_cleaned.csv"
OUT_XLSX = OUTPUT_DIR / "retail_sales_summary_tables.xlsx"

REQUIRED_COLUMNS = list(COL.values())
ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

## File handling helpers
Validates the input path, tries several encodings, checks required columns, and writes outputs safely (temp file then rename for CSV, skips Excel with a warning if `openpyxl` is missing).

In [34]:
def load_csv(path: Path) -> pd.DataFrame:
    """Load a CSV safely: validate path, try several encodings, check required columns."""
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path.resolve()}")
    if path.suffix.lower() not in (".csv", ".txt"):
        raise ValueError(f"Expected a .csv file, got '{path.suffix}'")
    if path.stat().st_size == 0:
        raise ValueError(f"Input file is empty: {path}")

    data = None
    last_error = None
    for enc in ENCODINGS:
        try:
            data = pd.read_csv(path, encoding=enc, sep=None, engine="python")
            log.info("Loaded %s with encoding=%s", path.name, enc)
            break
        except UnicodeDecodeError as e:
            last_error = e
        except pd.errors.EmptyDataError:
            raise ValueError(f"No columns found in {path}")
        except pd.errors.ParserError as e:
            raise ValueError(f"Could not parse {path}: {e}")
    if data is None:
        raise ValueError(f"Could not decode {path} with any of {ENCODINGS}") from last_error

    data.columns = data.columns.str.strip()
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}. "
                       f"Update the COL dictionary to match your file's headers.")
    if data.empty:
        raise ValueError("The CSV has headers but no data rows.")

    # Numeric columns sometimes arrive as text ("$1,200", "N/A"): strip symbols, then coerce.
    for key in ("quantity", "unit_price", "revenue"):
        col = COL[key]
        if not pd.api.types.is_numeric_dtype(data[col]):
            n_before = data[col].notna().sum()
            cleaned = data[col].astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
            data[col] = pd.to_numeric(cleaned, errors="coerce")
            lost = n_before - data[col].notna().sum()
            if lost:
                log.warning("%s: %d non-numeric values set to NaN", col, lost)
    return data


def save_csv(data: pd.DataFrame, path: Path) -> None:
    """Write CSV, creating the output folder and writing to a temp file first."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    data.to_csv(tmp, index=False, encoding="utf-8")
    tmp.replace(path)
    log.info("Saved %s (%d rows)", path, len(data))


def save_excel(tables: dict, path: Path) -> None:
    """Write several DataFrames to one workbook; skip gracefully if openpyxl is missing."""
    try:
        import openpyxl  # noqa: F401
    except ImportError:
        log.warning("openpyxl not installed; skipping Excel export. Run: pip install openpyxl")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for sheet, table in tables.items():
            table.to_excel(xw, sheet_name=sheet[:31])
    log.info("Saved %s (%d sheets)", path, len(tables))

## Load the data

In [35]:
df = load_csv(DATA_PATH)
df.head()

,Order_ID,Order_Date,City,Product,Category,Quantity,Unit_Price,Customer_ID,Payment_Mode,Revenue
0,ORD00000,14-02-2025,Bengaluru,Tablet,Computing,4,24999,C0125,UPI,99996
1,ORD00001,22-09-2025,Mumbai,Printer,Office Equipment,4,12999,C0023,Cash,51996
2,ORD00002,27-07-2025,Mumbai,Power Bank,Accessories,1,2499,C0148,Cash,2499
3,ORD00003,19-03-2026,Delhi,Power Bank,Accessories,1,1299,C0149,Card,1299
4,ORD00004,18-08-2025,Bengaluru,Tablet,Computing,4,24999,C0160,Card,99996


## Task 1: Dimensions, columns and data types

In [36]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
df.dtypes

Shape (rows, columns): (502, 10)

Columns: ['Order_ID', 'Order_Date', 'City', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Customer_ID', 'Payment_Mode', 'Revenue']

Data types:


,0
Order_ID,object
Order_Date,object
City,object
Product,object
Category,object
Quantity,int64
Unit_Price,int64
Customer_ID,object
Payment_Mode,object
Revenue,int64


## Task 2: Missing values and duplicate orders

In [52]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing"))

n_dup_rows = df.duplicated().sum()
n_dup_orders = df.duplicated(subset=[COL["order_id"]]).sum()
print(f"Fully duplicated rows: {n_dup_rows}")
print(f"Duplicate Order_IDs:   {n_dup_orders}")

df = df.drop_duplicates()
df = df.drop_duplicates(subset=[COL["order_id"]], keep="first")
print(f"Rows after removing duplicates: {len(df):,}")

Missing values per column:


,missing
Order_ID,0
Order_Date,0
City,0
Product,0
Category,0
Quantity,0
Unit_Price,0
Customer_ID,0
Payment_Mode,0
Revenue,0


Fully duplicated rows: 0
Duplicate Order_IDs:   0
Rows after removing duplicates: 500


## Task 3: Clean missing City values
**Strategy:** fill each missing city with the customer's most frequent known city. If the customer has no known city, label it `"Unknown"`. This keeps every transaction, avoids inventing locations, and makes the gap visible in later reports.

In [38]:
missing_before = df[COL["city"]].isna().sum()

if missing_before:
    cust_city = (
        df.dropna(subset=[COL["city"]])
          .groupby(COL["customer"])[COL["city"]]
          .agg(lambda s: s.mode().iat[0])
    )
    df[COL["city"]] = df[COL["city"]].fillna(df[COL["customer"]].map(cust_city))
    df[COL["city"]] = df[COL["city"]].fillna("Unknown")

print(f"Missing City before: {missing_before}")
print(f"Missing City after:  {df[COL['city']].isna().sum()}")

Missing City before: 10
Missing City after:  0


## Task 4: Convert Order_Date and extract month and weekday

In [39]:
df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")
print("Unparseable dates:", df[COL["date"]].isna().sum())

df["Month"] = df[COL["date"]].dt.month
df["Month_Name"] = df[COL["date"]].dt.strftime("%b")
df["Weekday"] = df[COL["date"]].dt.day_name()

df[[COL["date"], "Month", "Month_Name", "Weekday"]].head()

Unparseable dates: 0


/tmp/ipykernel_791/2193645757.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")


,Order_Date,Month,Month_Name,Weekday
0,2025-02-14,2,Feb,Friday
1,2025-09-22,9,Sep,Monday
2,2025-07-27,7,Jul,Sunday
3,2026-03-19,3,Mar,Thursday
4,2025-08-18,8,Aug,Monday


## Task 5: Calculate row-wise revenue and verify the supplied Revenue column

In [40]:
df["Calc_Revenue"] = df[COL["quantity"]] * df[COL["unit_price"]]
comparable = df["Calc_Revenue"].notna() & df[COL["revenue"]].notna()
mismatch = comparable & ~np.isclose(df["Calc_Revenue"], df[COL["revenue"]], atol=0.01)
print(f"Rows that could not be compared (missing value): {(~comparable).sum()}")

print(f"Rows where Revenue != Quantity x Unit_Price: {mismatch.sum()}")
if mismatch.any():
    cols = [COL["order_id"], COL["quantity"], COL["unit_price"], COL["revenue"], "Calc_Revenue"]
    display(df.loc[mismatch, cols].head(10))

Rows that could not be compared (missing value): 0
Rows where Revenue != Quantity x Unit_Price: 0


## Task 6: Total revenue, total quantity and average order value

In [41]:
total_rev = df[COL["revenue"]].sum()
total_qty = df[COL["quantity"]].sum()
n_orders = df[COL["order_id"]].nunique()
aov = total_rev / n_orders

summary = pd.Series({
    "Total revenue": total_rev,
    "Total quantity": total_qty,
    "Number of orders": n_orders,
    "Average order value": aov,
}, name="value")
summary.to_frame()

,value
Total revenue,"18,290,595.00"
Total quantity,"1,505.00"
Number of orders,500.00
Average order value,"36,581.19"


## Task 7: Top 10 products by revenue

In [42]:
top_products = (
    df.groupby(COL["product"])
      .agg(Revenue=(COL["revenue"], "sum"),
           Units=(COL["quantity"], "sum"),
           Orders=(COL["order_id"], "size"))
      .sort_values("Revenue", ascending=False)
      .head(10)
)
top_products

,Revenue,Units,Orders
Product,,,
Tablet,7325745,255,84
Smartwatch,4710237,263,93
External SSD,3384312,188,70
Bluetooth Speaker,1259523,277,82
Printer,1239279,221,73
Power Bank,371499,301,98


## Task 8: Cities ranked by revenue

In [43]:
city_rank = (
    df.groupby(COL["city"])
      .agg(Revenue=(COL["revenue"], "sum"),
           Orders=(COL["order_id"], "size"))
      .sort_values("Revenue", ascending=False)
)
city_rank["Rank"] = np.arange(1, len(city_rank) + 1)
city_rank

,Revenue,Orders,Rank
City,,,
Hyderabad,4683663,116,1
Bengaluru,3706194,94,2
Pune,3564329,90,3
Delhi,3361258,111,4
Mumbai,2975151,89,5


## Task 9: Monthly revenue and best month

In [44]:
monthly = (
    df.groupby("Month")
      .agg(Revenue=(COL["revenue"], "sum"),
           Orders=(COL["order_id"], "size"))
)
best_month = monthly["Revenue"].idxmax()
print(f"Best month: {best_month} with revenue {monthly.loc[best_month, 'Revenue']:,.2f}")
monthly

Best month: 5 with revenue 2,405,593.00


,Revenue,Orders
Month,,
1,1942735,56
2,2361124,52
3,1854624,63
4,1087503,34
5,2405593,63
6,1163463,47
7,1015736,23
8,1638189,39
9,1208721,27


## Task 10: Top 10 customers by total spending

In [45]:
top_customers = (
    df.groupby(COL["customer"])
      .agg(Spend=(COL["revenue"], "sum"),
           Orders=(COL["order_id"], "size"))
      .sort_values("Spend", ascending=False)
      .head(10)
)
top_customers

,Spend,Orders
Customer_ID,,
C0095,603984,4
C0106,537480,5
C0098,524985,3
C0132,371188,3
C0028,366470,10
C0108,361674,6
C0096,357984,5
C0107,353983,4
C0026,350991,4


## Task 11: Payment methods by orders and revenue

In [46]:
payment = (
    df.groupby(COL["payment"])
      .agg(Orders=(COL["order_id"], "size"),
           Revenue=(COL["revenue"], "sum"))
      .sort_values("Revenue", ascending=False)
)
payment["Share_of_Revenue_%"] = payment["Revenue"] / total_rev * 100
payment

,Orders,Revenue,Share_of_Revenue_%
Payment_Mode,,,
UPI,175,6280960,34.34
Card,110,4594291,25.12
Cash,113,4326464,23.65
NetBanking,102,3088880,16.89


## Task 12: Unusually large order values (IQR method)
Outliers are values above `Q3 + 1.5 × IQR` or below `Q1 − 1.5 × IQR`. Quartiles are calculated with `np.percentile`.

In [47]:
rev = df[COL["revenue"]].dropna().to_numpy()
q1, q3 = np.percentile(rev, [25, 75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

outliers = df[(df[COL["revenue"]] > upper) | (df[COL["revenue"]] < lower)]

print(f"Q1={q1:,.2f}  Q3={q3:,.2f}  IQR={iqr:,.2f}")
print(f"Upper fence={upper:,.2f}  Lower fence={lower:,.2f}")
print(f"Outlier orders: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}% of orders)")
print(f"Outlier revenue share: {outliers[COL['revenue']].sum() / total_rev * 100:.2f}%")
outliers.sort_values(COL["revenue"], ascending=False).head(10)

Q1=4,998.00  Q3=46,998.75  IQR=42,000.75
Upper fence=109,999.88  Lower fence=-58,003.12
Outlier orders: 44 (8.80% of orders)
Outlier revenue share: 44.48%


,Order_ID,Order_Date,City,Product,Category,Quantity,Unit_Price,Customer_ID,Payment_Mode,Revenue,Month,Month_Name,Weekday,Calc_Revenue
51,ORD00051,2026-03-31,Delhi,Tablet,Computing,5,45999,C0012,Card,229995,3,Mar,Tuesday,229995
50,ORD00050,2025-10-02,Hyderabad,External SSD,Storage,5,45999,C0045,UPI,229995,10,Oct,Thursday,229995
69,ORD00069,2026-01-08,Pune,Tablet,Computing,5,45999,C0106,UPI,229995,1,Jan,Thursday,229995
68,ORD00068,2025-07-29,Bengaluru,Smartwatch,Wearables,5,45999,C0107,Card,229995,7,Jul,Tuesday,229995
76,ORD00076,2025-11-13,Mumbai,External SSD,Storage,5,45999,C0086,Cash,229995,11,Nov,Thursday,229995
102,ORD00102,2025-02-10,Delhi,Smartwatch,Wearables,5,45999,C0051,UPI,229995,2,Feb,Monday,229995
300,ORD00300,2025-11-06,Pune,External SSD,Storage,5,45999,C0077,Card,229995,11,Nov,Thursday,229995
298,ORD00298,2026-01-21,Mumbai,Smartwatch,Wearables,5,45999,C0140,Cash,229995,1,Jan,Wednesday,229995
184,ORD00184,2025-07-27,Hyderabad,Smartwatch,Wearables,5,45999,C0098,UPI,229995,7,Jul,Sunday,229995
211,ORD00211,2026-05-11,Delhi,External SSD,Storage,5,45999,C0098,Card,229995,5,May,Monday,229995


## Task 13: Product-category summary using groupby

In [48]:
category = (
    df.groupby(COL["category"])
      .agg(Orders=(COL["order_id"], "size"),
           Units=(COL["quantity"], "sum"),
           Revenue=(COL["revenue"], "sum"),
           Avg_Order_Value=(COL["revenue"], "mean"))
      .sort_values("Revenue", ascending=False)
)
category.round(2)

,Orders,Units,Revenue,Avg_Order_Value
Category,,,,
Computing,84,255,7325745,"87,211.25"
Wearables,93,263,4710237,"50,647.71"
Storage,70,188,3384312,"48,347.31"
Audio,82,277,1259523,"15,360.04"
Office Equipment,73,221,1239279,"16,976.42"
Accessories,98,301,371499,"3,790.81"


## Task 14: Mean, median and standard deviation of order revenue (NumPy)

In [49]:
rev = df[COL["revenue"]].dropna().to_numpy()
print(f"Mean:                 {np.mean(rev):,.2f}")
print(f"Median:               {np.median(rev):,.2f}")
print(f"Std dev (sample):     {np.std(rev, ddof=1):,.2f}")

Mean:                 36,581.19
Median:               12,999.00
Std dev (sample):     54,015.71


## Task 15: Business recommendations
Write your five recommendations in the markdown cell below, citing the numbers from Tasks 6–14. Run the cells above first so the figures are current.

1.
2.
3.
4.
5.

## Extra: Weekday revenue pattern

In [50]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = (
    df.groupby("Weekday")[COL["revenue"]].sum()
      .reindex(weekday_order)
)
weekday

,Revenue
Weekday,
Monday,2499694
Tuesday,2405103
Wednesday,3150995
Thursday,2580178
Friday,2777845
Saturday,2946903
Sunday,1929877


## Save outputs

In [51]:
try:
    save_csv(df.drop(columns=["Calc_Revenue"]), OUT_CLEAN)
    save_excel({
        "Top_Products": top_products,
        "City_Ranking": city_rank,
        "Monthly": monthly,
        "Top_Customers": top_customers,
        "Payment_Methods": payment,
        "Category_Summary": category,
    }, OUT_XLSX)
except OSError as e:
    log.error("Could not write output files: %s", e)
    raise

print(f"Outputs written to: {OUTPUT_DIR.resolve()}")

Outputs written to: /content/outputs
